In [0]:
import requests

# Consulto a página oficial ANP onde são publicados os levantamentos semanis de preços por posto revendedor.

url_anp = (
    "https://www.gov.br/anp/pt-br/assuntos/"
    "precos-e-defesa-da-concorrencia/precos/"
    "levantamento-de-precos-de-combustiveis-ultimas-semanas-pesquisadas"
)

resposta = requests.get(
    url_anp,
    timeout=30
)
resposta.raise_for_status()

print("Status da página:", resposta.status_code)
print("Tamanho do conteúdo:", len(resposta.text))

In [0]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Analiso o HTML retornado pela página oficial da ANP.
sopa = BeautifulSoup(
    resposta.text,
    "html.parser"
)

# Procuro os links referentes aos preços por posto revendedor.
links_postos = []

for link in sopa.find_all("a", href=True):
    texto_link = link.get_text(
        " ",
        strip=True
    )

    if "Preços por posto revendedor" in texto_link:
        links_postos.append({
            "texto": texto_link,
            "url": urljoin(
                url_anp,
                link["href"]
            )
        })

if not links_postos:
    raise ValueError(
        "Nenhum link de preços por posto revendedor foi encontrado."
    )

arquivo_mais_recente = links_postos[0]

print("Links encontrados:", len(links_postos))
print("Arquivo mais recente:")
print(arquivo_mais_recente["texto"])
print(arquivo_mais_recente["url"])

In [0]:
import os

# Baixo o arquivo mais recente encontrado na página da ANP, mas ainda não salvo nada no Volume.
url_arquivo = arquivo_mais_recente["url"]

resposta_arquivo = requests.get(
    url_arquivo,
    timeout=60
)

resposta_arquivo.raise_for_status()

nome_arquivo = os.path.basename(url_arquivo)

print("Status do download:", resposta_arquivo.status_code)
print("Arquivo:", nome_arquivo)
print("Content-Type:", resposta_arquivo.headers.get("Content-Type"))
print("Tamanho em bytes:", len(resposta_arquivo.content))
print("Assinatura inicial:", resposta_arquivo.content[:2])

In [0]:
import hashlib
import os

# Calculo o SHA-256 diretamente do conteúdo baixado.
hash_arquivo = hashlib.sha256(
    resposta_arquivo.content
).hexdigest()

# Organizo a RAW por hash para manter cada arquivo identificado de forma única e evitar sobrescritas.
pasta_destino = (
    f"/Volumes/workspace/raw/anp_arquivos/anp/{hash_arquivo}"
)

caminho_destino = os.path.join(
    pasta_destino,
    nome_arquivo
)

os.makedirs(
    pasta_destino,
    exist_ok=True
)

# Gravo diretamente no Volume do Databricks.
if os.path.exists(caminho_destino):
    print("Arquivo já existe na RAW.")
else:
    with open(caminho_destino, "wb") as arquivo:
        arquivo.write(resposta_arquivo.content)

    print("Arquivo salvo na RAW.")

print("SHA-256:")
print(hash_arquivo)

print("Caminho RAW:")
print(caminho_destino)

In [0]:
# Disponibilizo o caminho do arquivo salvo na RAW para que a próxima tarefa do Job possa utilizá-lo.
dbutils.jobs.taskValues.set(
    key="caminho_arquivo",
    value=caminho_destino
)

print("Caminho disponibilizado para a próxima tarefa:")
print(caminho_destino)